# NSE Shareholding Downloader (Colab)
Downloads **~83,173** shareholding documents from NSE → Google Drive.

| Type  | Count   | Action |
|-------|---------|--------|
| XML   | ~63,071 | Saved as-is |
| PDF   | ~15,375 | Saved as-is |
| XLSX  | ~3,066  | Saved as-is |
| ZIP   | ~1,661  | PDFs extracted |
| Junk URLs (`xbrl/-`, `xbrl/null`) | ~58,742 | Auto-skipped |

- Full resume support — re-run any time, nothing re-downloaded
- Manifest saved every 100 files **and** every 4 minutes

**Run Cell 0 → 1 → 2 → 3 → 4**

In [1]:
# CELL 0: Keepalive — prevents Colab from disconnecting mid-run
import threading, time
def _ka():
    while True: time.sleep(55); _ = 1 + 1
if not any(t.name == 'keepalive' for t in threading.enumerate()):
    threading.Thread(target=_ka, name='keepalive', daemon=True).start()
    print('✅ Keepalive started')
else:
    print('✅ Keepalive already running')

✅ Keepalive started


In [2]:
# CELL 1: Mount Drive (skips if already mounted)
import os, shutil, time
from google.colab import drive
MOUNT = '/content/drive'
already_ok = False
try:
    tp = os.path.join(MOUNT, 'MyDrive')
    if os.path.isdir(tp) and os.listdir(tp):
        already_ok = True
except:
    pass
if already_ok:
    print('✅ Drive already mounted — skipping.')
else:
    if os.path.isdir(MOUNT):
        try: shutil.rmtree(MOUNT, ignore_errors=True)
        except: pass
    for attempt in range(1, 6):
        try:
            drive.mount(MOUNT)
            print('✅ Drive mounted!')
            break
        except Exception as e:
            if 'already contain' in str(e): shutil.rmtree(MOUNT, ignore_errors=True)
            if attempt < 5:
                print(f'Mount {attempt}/5 failed: {e}, retrying in {10*attempt}s')
                time.sleep(10 * attempt)
            else:
                raise

Mounted at /content/drive
✅ Drive mounted!


In [3]:
# CELL 2: Set paths — hardcoded for instant startup (no slow os.walk)
import os

# ── Paths (match your Drive layout exactly) ───────────────────────────────────
MANIFEST_CSV = '/content/drive/MyDrive/ArthLLM-2B/shareholding/nse_shareholding.csv'
# Output: shareholding/NSE_Shareholding_Docs/raw_files/<SYMBOL>/

if not os.path.exists(MANIFEST_CSV):
    raise FileNotFoundError(
        f'CSV not found:\n  {MANIFEST_CSV}\n'
        'Check your Drive path — folder is "shareholding" (lowercase).'
    )

BASE    = os.path.dirname(MANIFEST_CSV)
OUT_DIR = os.path.join(BASE, 'NSE_Shareholding_Docs', 'raw_files')
os.makedirs(OUT_DIR, exist_ok=True)

print(f'✅ Manifest : {MANIFEST_CSV}')
print(f'📥 Output   : {OUT_DIR}')

✅ Manifest : /content/drive/MyDrive/ArthLLM-2B/shareholding/nse_shareholding.csv
📥 Output   : /content/drive/MyDrive/ArthLLM-2B/shareholding/NSE_Shareholding_Docs/raw_files


In [4]:
# CELL 3: Fast Resume — two-way sync manifest ↔ disk
import os, re
import pandas as pd
from tqdm.notebook import tqdm

# ── Load CSV ──────────────────────────────────────────────────────────────────
fsize = os.path.getsize(MANIFEST_CSV) if os.path.exists(MANIFEST_CSV) else 0
if fsize == 0: raise RuntimeError('CORRUPT CSV: 0 bytes')
try:
    df = pd.read_csv(MANIFEST_CSV, dtype=str, low_memory=False)
except Exception as e:
    raise RuntimeError(f'CORRUPT CSV: {e}')
if len(df) == 0: raise RuntimeError('CSV has 0 rows')
print(f'Loaded {len(df):,} rows')

# ── Normalise dl_status column ────────────────────────────────────────────────
DONE_VALS = ('done', 'skip', 'true')
if 'dl_status' not in df.columns:
    df['dl_status'] = 'pending'
else:
    df['dl_status'] = df['dl_status'].apply(
        lambda v: 'done' if str(v).strip().lower() in DONE_VALS
                  else (str(v).strip().lower() if pd.notna(v) and str(v).strip() != '' else 'pending')
    )

# ── Mark junk / missing URLs — never attempt these ───────────────────────────
# Junk patterns: NaN, '-', '', 'nan', ends with '/null', ends with '/-'
def is_junk_url(u):
    s = str(u).strip()
    if s in ('', '-', 'nan'): return True
    if s.endswith('/null') or s.endswith('/-'): return True
    return False

junk_mask = df['pdf_url'].apply(is_junk_url)
df.loc[junk_mask, 'dl_status'] = 'no_url'
print(f'  Junk/missing URLs skipped : {junk_mask.sum():,}')

# ── File types we save to disk ────────────────────────────────────────────────
KEEP_EXTS = {'.pdf', '.xml', '.xlsx', '.rar'}

# ── Scan existing files on disk ───────────────────────────────────────────────
print('Scanning disk for existing files...')
existing = set()
if os.path.exists(OUT_DIR):
    for root, dirs, files in os.walk(OUT_DIR):
        for x in files:
            if os.path.splitext(x)[1].lower() in KEEP_EXTS | {'.pdf'}:
                existing.add(x)
print(f'Found {len(existing):,} files already on disk')

# ── Helper: derive expected filename for a row ────────────────────────────────
def safe(s, mx=80):
    return re.sub(r'[<>:"/\\|?*\s]+', '_', str(s).strip())[:mx]

def expected_fname(row):
    sym  = safe(row.get('symbol', 'UNKNOWN'))
    seq  = safe(str(row.get('seq_id', '')).split('.')[0])
    url  = str(row.get('pdf_url', ''))
    ext  = os.path.splitext(url)[1].lower()
    base = os.path.splitext(os.path.basename(url))[0][:50]
    # ZIPs produce .pdf; everything else keeps original extension
    out_ext = '.pdf' if ext == '.zip' else (ext if ext in KEEP_EXTS else ext)
    return f'{sym}_{seq}_{base}{out_ext}'

# ── Two-way sync: manifest ↔ disk ─────────────────────────────────────────────
updated = 0
for i in tqdm(df.index, desc='Syncing manifest'):
    if df.at[i, 'dl_status'] == 'no_url': continue
    cur = df.at[i, 'dl_status']
    fn  = expected_fname(df.loc[i])
    fe  = fn in existing
    if cur == 'done' and not fe:  df.at[i, 'dl_status'] = 'pending'; updated += 1
    elif cur != 'done' and fe:    df.at[i, 'dl_status'] = 'done';    updated += 1

if updated > 0:
    df.to_csv(MANIFEST_CSV, index=False, quoting=1)
    print(f'Updated {updated:,} rows in manifest')
else:
    print('Manifest already in sync — no changes needed')

print('\nStatus summary:')
for s, c in df['dl_status'].value_counts().items():
    print(f'  {s:15s}: {c:,}')

Loaded 141,915 rows
  Junk/missing URLs skipped : 58,742
Scanning disk for existing files...
Found 0 files already on disk


Syncing manifest:   0%|          | 0/141915 [00:00<?, ?it/s]

Manifest already in sync — no changes needed

Status summary:
  pending        : 83,173
  no_url         : 58,742


In [ ]:
# CELL 4: Download NSE Shareholding Docs
# Handles: .pdf | .xml | .xlsx | .zip (extract PDFs) | skips junk URLs
import pandas as pd, requests, os, re, time, threading, zipfile, io
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

# ── Tuning ────────────────────────────────────────────────────────────────────
WORKERS     = 12
SAVE_EVERY  = 100
TIMER_SAVE  = 240
TIMEOUT     = 45
MAX_RETRIES = 5
RETRY_CODES = {429, 500, 502, 503, 504}
KEEP_EXTS   = {'.pdf', '.xml', '.xlsx', '.rar'}

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    ),
    'Accept'          : 'application/pdf,application/xml,application/zip,application/octet-stream,*/*',
    'Accept-Encoding' : 'gzip, deflate, br',
    'Connection'      : 'keep-alive',
    'Referer'         : 'https://www.nseindia.com/',
}

# ── Helpers ───────────────────────────────────────────────────────────────────
def safe(s, mx=80):
    return re.sub(r'[<>:"/\\|?*\s]+', '_', str(s).strip())[:mx]

def is_junk_url(u):
    s = str(u).strip()
    if s in ('', '-', 'nan'): return True
    if s.endswith('/null') or s.endswith('/-'): return True
    return False

def is_html_error(data: bytes) -> bool:
    if not data: return True
    head = data[:1024].lower()
    return b'<html' in head or b'<!doctype' in head

def is_valid_pdf(data: bytes) -> bool:
    if not data or len(data) < 64: return False
    if data[:4] == b'%PDF': return True
    if is_html_error(data): return False
    return len(data) > 4096

def extract_pdfs_from_zip(zip_bytes: bytes, dest_dir: str, base_name: str) -> list:
    """Extract all PDFs from a ZIP (one level of nested ZIPs too)."""
    saved = []
    try:
        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
            members     = zf.namelist()
            pdf_members = [m for m in members if m.lower().endswith('.pdf')]
            if not pdf_members:
                for m in members:
                    if m.lower().endswith('.zip'):
                        try: saved += extract_pdfs_from_zip(zf.read(m), dest_dir, base_name + '_inner')
                        except: pass
                # Save any XML/XLSX inside ZIP too
                for m in members:
                    if os.path.splitext(m)[1].lower() in ('.xml', '.xlsx'):
                        try:
                            fname = f'{base_name}_{os.path.basename(m)}'
                            os.makedirs(dest_dir, exist_ok=True)
                            with open(os.path.join(dest_dir, fname), 'wb') as f:
                                f.write(zf.read(m))
                            saved.append(fname)
                        except: pass
                return saved
            os.makedirs(dest_dir, exist_ok=True)
            for idx, member in enumerate(pdf_members):
                try:
                    suffix = f'_{idx+1}' if len(pdf_members) > 1 else ''
                    fname  = f'{base_name}{suffix}.pdf'
                    with open(os.path.join(dest_dir, fname), 'wb') as f:
                        f.write(zf.read(member))
                    saved.append(fname)
                except: pass
    except: pass
    return saved

def _get_with_retry(url: str):
    """GET with exponential back-off. Returns Response or None."""
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            if r.status_code == 200: return r
            if r.status_code in RETRY_CODES:
                time.sleep(min(60, 5 * (2 ** attempt)))
                continue
            return r  # 404/403 — no point retrying
        except requests.exceptions.Timeout:
            time.sleep(4 * (attempt + 1))
        except requests.exceptions.ConnectionError:
            time.sleep(6 * (attempt + 1))
        except Exception:
            time.sleep(3 * (attempt + 1))
    return None

def download_row(url: str, dest_dir: str, base_name: str) -> str:
    """
    Download one row. Handles PDF / XML / XLSX / ZIP.
    Returns: 'skip' | 'ok' | 'fail' | 'bad_content' | 'http_NNN'
    """
    if is_junk_url(url):
        return 'skip'  # safety net (manifest should have caught these)

    ext      = os.path.splitext(url)[1].lower()  # .pdf / .xml / .xlsx / .zip / ''
    is_zip   = (ext == '.zip')
    out_ext  = '.pdf' if is_zip else (ext if ext else '.bin')
    dest_file = os.path.join(dest_dir, f'{base_name}{out_ext}')

    # ── Skip check ────────────────────────────────────────────────────────────
    if is_zip:
        if (os.path.exists(dest_file) or
                os.path.exists(os.path.join(dest_dir, f'{base_name}_1.pdf'))):
            return 'skip'
    else:
        if os.path.exists(dest_file) and os.path.getsize(dest_file) > 512:
            return 'skip'

    r = _get_with_retry(url)
    if r is None: return 'fail'
    if r.status_code != 200: return f'http_{r.status_code}'

    data = r.content

    # HTML error page check (applies to all types)
    if ext not in ('.xml',) and is_html_error(data):
        return 'bad_content'

    os.makedirs(dest_dir, exist_ok=True)

    if is_zip:
        saved = extract_pdfs_from_zip(data, dest_dir, base_name)
        return 'ok' if saved else 'fail'

    if ext == '.pdf' and not is_valid_pdf(data):
        return 'bad_content'

    with open(dest_file, 'wb') as f:
        f.write(data)
    return 'ok'

# ── Load manifest ─────────────────────────────────────────────────────────────
fsize = os.path.getsize(MANIFEST_CSV) if os.path.exists(MANIFEST_CSV) else 0
if fsize == 0: raise RuntimeError('CORRUPT CSV: 0 bytes')
try:
    df = pd.read_csv(MANIFEST_CSV, dtype=str, low_memory=False)
except Exception as e:
    raise RuntimeError(f'Cannot read CSV: {e}')

DONE_VALS = ('done', 'skip', 'true')
if 'dl_status' not in df.columns:
    df['dl_status'] = 'pending'
else:
    df['dl_status'] = df['dl_status'].apply(
        lambda v: 'done' if str(v).strip().lower() in DONE_VALS
                  else (str(v).strip().lower() if pd.notna(v) and str(v).strip() != '' else 'pending')
    )

# Mark all junk URLs
junk_mask = df['pdf_url'].apply(is_junk_url)
df.loc[junk_mask, 'dl_status'] = 'no_url'

# ── Build task list ───────────────────────────────────────────────────────────
pending_df = df[~df['dl_status'].isin(['done', 'no_url'])]
print(f'Pending : {len(pending_df):,}  |  '
      f'Done : {(df["dl_status"]=="done").sum():,}  |  '
      f'Junk/No URL : {junk_mask.sum():,}  |  '
      f'Total : {len(df):,}')

if len(pending_df) == 0:
    print('🎉 All done! Nothing left to download.')
else:
    tasks = []
    for idx, row in pending_df.iterrows():
        url       = str(row['pdf_url']).strip()
        sym       = safe(row.get('symbol', 'UNKNOWN'))
        seq       = safe(str(row.get('seq_id', '')).split('.')[0])
        url_base  = os.path.splitext(os.path.basename(url))[0][:50]
        base_name = f'{sym}_{seq}_{url_base}'
        dest_dir  = os.path.join(OUT_DIR, sym)
        tasks.append((idx, url, dest_dir, base_name))

    ok = skip = fail = counter = 0
    _stop = threading.Event()
    _lock = threading.Lock()

    def _autosave():
        while not _stop.wait(TIMER_SAVE):
            with _lock:
                try: df.to_csv(MANIFEST_CSV, index=False, quoting=1)
                except: pass
    threading.Thread(target=_autosave, daemon=True, name='autosave').start()

    try:
        with ThreadPoolExecutor(max_workers=WORKERS) as pool:
            futs = {
                pool.submit(download_row, u, d, b): (i, u)
                for i, u, d, b in tasks
            }
            pbar = tqdm(total=len(futs), desc='NSE Shareholding Docs', unit='file')
            for fut in as_completed(futs):
                i, u = futs[fut]
                try:    res = fut.result()
                except: res = 'fail'

                if res in ('ok', 'skip'):
                    df.at[i, 'dl_status'] = 'done'
                    if res == 'ok': ok   += 1
                    else:           skip += 1
                else:
                    fail += 1
                    df.at[i, 'dl_status'] = res

                counter += 1
                if counter % SAVE_EVERY == 0:
                    with _lock:
                        df.to_csv(MANIFEST_CSV, index=False, quoting=1)

                pbar.set_postfix(ok=ok, skip=skip, fail=fail, refresh=False)
                pbar.update(1)
            pbar.close()

    except KeyboardInterrupt:
        print('\n⚠️  Interrupted — saving manifest before exit...')
    finally:
        _stop.set()
        with _lock:
            df.to_csv(MANIFEST_CSV, index=False, quoting=1)
        print(f'💾 Manifest saved → {MANIFEST_CSV}')

    total     = len(df)
    done_n    = (df['dl_status'] == 'done').sum()
    no_url_n  = (df['dl_status'] == 'no_url').sum()
    pending_n = total - done_n - no_url_n

    sep = '=' * 52
    print(f'\n{sep}')
    print(f'  SUMMARY')
    print(f'  Total rows      : {total:,}')
    print(f'  Done (total)    : {done_n:,}')
    print(f'    └ New OK      : {ok:,}')
    print(f'    └ Skipped     : {skip:,}')
    print(f'  Failed          : {fail:,}')
    print(f'  Junk/No URL     : {no_url_n:,}')
    print(f'  Still pending   : {pending_n:,}')
    print(f'{sep}')
    if fail > 0:
        print(f'\n💡 Tip: Re-run Cell 4 to retry the {fail:,} failed files.')
        failed_codes = df[
            ~df['dl_status'].isin(['done', 'no_url', 'pending'])
        ]['dl_status'].value_counts()
        print('  Failure breakdown:')
        for code, cnt in failed_codes.items():
            print(f'    {code}: {cnt:,}')